In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv("ALPACA_API_KEY")
secret_key = os.getenv("ALPACA_SECRET_KEY")

print("Keys cargadas:", api_key is not None, secret_key is not None)

Keys cargadas: True True


In [2]:
!pip install alpaca-py


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
from alpaca.data.historical import StockHistoricalDataClient
from alpaca.data.requests import StockBarsRequest
from alpaca.data.timeframe import TimeFrame
from datetime import datetime

In [5]:
client = StockHistoricalDataClient(api_key, secret_key)

In [6]:
request_params = StockBarsRequest(
    symbol_or_symbols="TSLA",
    timeframe=TimeFrame.Day,
    start=datetime(2020, 1, 1),
    end=datetime(2026, 7, 21)
)

bars = client.get_stock_bars(request_params)
df = bars.df

print(df.shape)
df.head()

(1644, 7)


open      high      low   close  \
symbol timestamp                                                      
TSLA   2020-01-02 05:00:00+00:00  424.50  430.6957  421.710  430.26   
       2020-01-03 05:00:00+00:00  440.50  454.0000  436.920  443.01   
       2020-01-06 05:00:00+00:00  440.47  451.5600  440.000  451.50   
       2020-01-07 05:00:00+00:00  461.40  471.6300  453.355  469.06   
       2020-01-08 05:00:00+00:00  473.70  498.4900  468.230  492.14   

                                      volume  trade_count        vwap  
symbol timestamp                                                       
TSLA   2020-01-02 05:00:00+00:00   9763456.0     136851.0  427.117489  
       2020-01-03 05:00:00+00:00  18034327.0     236310.0  446.323497  
       2020-01-06 05:00:00+00:00  10362505.0     134223.0  445.863261  
       2020-01-07 05:00:00+00:00  18488930.0     250189.0  464.051044  
       2020-01-08 05:00:00+00:00  31486817.0     413644.0  485.693897

In [7]:
df.to_csv("../data/raw/tsla_raw.csv")
print("Guardado correctamente")

Guardado correctamente


In [8]:

df = df.sort_index()

# 1. Retorno diario (% de cambio respecto al día anterior)
df['retorno_diario'] = df['close'].pct_change()

# 2. Media móvil de 5 días
df['sma_5'] = df['close'].rolling(window=5).mean()

# 3. Media móvil de 20 días
df['sma_20'] = df['close'].rolling(window=20).mean()

# 4. Volatilidad (desviación estándar de 5 días)
df['volatilidad_5'] = df['close'].rolling(window=5).std()

# 5. Volumen promedio de 5 días
df['volumen_promedio_5'] = df['volume'].rolling(window=5).mean()

# 6. Precio de cierre del día anterior (lag)
df['precio_anterior'] = df['close'].shift(1)

# Variable objetivo: precio de cierre del día SIGUIENTE
df['precio_futuro'] = df['close'].shift(-1)

print(df.shape)
df.head(10)

(1644, 14)


open      high       low   close  \
symbol timestamp                                                        
TSLA   2020-01-02 05:00:00+00:00  424.500  430.6957  421.7100  430.26   
       2020-01-03 05:00:00+00:00  440.500  454.0000  436.9200  443.01   
       2020-01-06 05:00:00+00:00  440.470  451.5600  440.0000  451.50   
       2020-01-07 05:00:00+00:00  461.400  471.6300  453.3550  469.06   
       2020-01-08 05:00:00+00:00  473.700  498.4900  468.2300  492.14   
       2020-01-09 05:00:00+00:00  497.100  498.8000  472.8700  481.34   
       2020-01-10 05:00:00+00:00  481.790  484.9400  473.7000  478.15   
       2020-01-13 05:00:00+00:00  493.500  525.6300  492.0000  524.86   
       2020-01-14 05:00:00+00:00  544.255  547.4100  524.9000  537.92   
       2020-01-15 05:00:00+00:00  529.760  537.8400  516.7853  518.50   

                                      volume  trade_count        vwap  \
symbol timestamp                                                        
TSLA   2020-01-02 05:00:00+00:00   9763456.0     136851.0  427.117489   
       2020-01-03 05:00:00+00:00  18034327.0     236310.0  446.323497   
       2020-01-06 05:00:00+00:00  10362505.0     134223.0  445.863261   
       2020-01-07 05:00:00+00:00  18488930.0     250189.0  464.051044   
       2020-01-08 05:00:00+00:00  31486817.0     413644.0  485.693897   
       2020-01-09 05:00:00+00:00  28959816.0     384282.0  485.533718   
       2020-01-10 05:00:00+00:00  13217075.0     181667.0  478.804229   
       2020-01-13 05:00:00+00:00  26916914.0     369915.0  511.288547   
       2020-01-14 05:00:00+00:00  29600289.0     406428.0  538.125989   
       2020-01-15 05:00:00+00:00  17643746.0     261961.0  529.061934   

                                  retorno_diario    sma_5  sma_20  \
symbol timestamp                                                    
TSLA   2020-01-02 05:00:00+00:00             NaN      NaN     NaN   
       2020-01-03 05:00:00+00:00        0.029633      NaN     NaN   
       2020-01-06 05:00:00+00:00        0.019164      NaN     NaN   
       2020-01-07 05:00:00+00:00        0.038893      NaN     NaN   
       2020-01-08 05:00:00+00:00        0.049205  457.194     NaN   
       2020-01-09 05:00:00+00:00       -0.021945  467.410     NaN   
       2020-01-10 05:00:00+00:00       -0.006627  474.438     NaN   
       2020-01-13 05:00:00+00:00        0.097689  489.110     NaN   
       2020-01-14 05:00:00+00:00        0.024883  502.882     NaN   
       2020-01-15 05:00:00+00:00       -0.036102  508.154     NaN   

                                  volatilidad_5  volumen_promedio_5  \
symbol timestamp                                                      
TSLA   2020-01-02 05:00:00+00:00            NaN                 NaN   
       2020-01-03 05:00:00+00:00            NaN                 NaN   
       2020-01-06 05:00:00+00:00            NaN                 NaN   
       2020-01-07 05:00:00+00:00            NaN                 NaN   
       2020-01-08 05:00:00+00:00      24.088756          17627207.0   
       2020-01-09 05:00:00+00:00      20.352079          21466479.0   
       2020-01-10 05:00:00+00:00      15.246738          20503028.6   
       2020-01-13 05:00:00+00:00      21.620247          23813910.4   
       2020-01-14 05:00:00+00:00      26.934244          26036182.2   
       2020-01-15 05:00:00+00:00      26.885756          23267568.0   

                                  precio_anterior  precio_futuro  
symbol timestamp                                                  
TSLA   2020-01-02 05:00:00+00:00              NaN         443.01  
       2020-01-03 05:00:00+00:00           430.26         451.50  
       2020-01-06 05:00:00+00:00           443.01         469.06  
       2020-01-07 05:00:00+00:00           451.50         492.14  
       2020-01-08 05:00:00+00:00           469.06         481.34  
       2020-01-09 05:00:00+00:00           492.14         478.15  
       2020-01-10 05:00:00+00:00           481.34         524.86 

In [9]:
# Ver cuántos nulos hay por columna
print(df.isnull().sum())

# Eliminar filas con valores nulos
df_limpio = df.dropna()

print("Antes:", df.shape)
print("Después de limpiar:", df_limpio.shape)

open                   0
high                   0
low                    0
close                  0
volume                 0
trade_count            0
vwap                   0
retorno_diario         1
sma_5                  4
sma_20                19
volatilidad_5          4
volumen_promedio_5     4
precio_anterior        1
precio_futuro          1
dtype: int64
Antes: (1644, 14)
Después de limpiar: (1624, 14)


In [10]:
from sklearn.preprocessing import StandardScaler

# Definimos qué columnas son nuestras features (X) y cuál es el objetivo (y)
features = ['open', 'high', 'low', 'close', 'volume', 'retorno_diario',
            'sma_5', 'sma_20', 'volatilidad_5', 'volumen_promedio_5', 'precio_anterior']

X = df_limpio[features]
y = df_limpio['precio_futuro']

# Escalar las features
scaler = StandardScaler()
X_escalado = scaler.fit_transform(X)

print("Forma de X escalado:", X_escalado.shape)
print("Forma de y:", y.shape)

Forma de X escalado: (1624, 11)
Forma de y: (1624,)


In [11]:
import pandas as pd


df_final = pd.DataFrame(X_escalado, columns=features)
df_final['precio_futuro'] = y.values


df_final.to_csv("../data/processed/tsla_processed.csv", index=False)

print("Dataset final guardado correctamente")
print(df_final.shape)
df_final.head()

Dataset final guardado correctamente
(1624, 12)


,open,high,low,close,volume,retorno_diario,sma_5,sma_20,volatilidad_5,volumen_promedio_5,precio_anterior,precio_futuro
0,0.429893,0.437281,0.436630,0.455516,-0.877272,2.114542,0.276856,0.091213,0.284992,-1.181532,0.270217,650.57
1,0.453347,0.443657,0.482922,0.485687,-1.147637,0.290142,0.330429,0.126750,0.480616,-1.175817,0.455141,780.00
2,0.557587,0.844040,0.613636,0.885793,-0.505550,4.110606,0.469113,0.181108,1.340841,-1.030915,0.485313,887.06
3,1.205094,1.393913,1.124888,1.216747,-0.224783,2.827700,0.669137,0.251366,2.161437,-0.818096,0.885425,734.70
4,1.020375,1.023993,0.711164,0.745757,-0.478977,-3.598348,0.765169,0.294215,1.695284,-0.686598,1.216384,748.96
